In [ ]:
!pip install pyspark

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("projetoanalisecinema").getOrCreate()

In [ ]:
# Leitura CSV Filme B

df_salas = spark.read.csv(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_FILMEB/Raking de estado por sala - Página1.csv",header=True,inferSchema=True)

df_publico = spark.read.csv(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_FILMEB/Ranking de estados por público - Filme B - Página1.csv",header=True,inferSchema=True)

In [ ]:
# Conferir CSV

df_salas.show(5, truncate=False)
df_salas.printSchema()

df_publico.show(5, truncate=False)
df_publico.printSchema()

+--------------+---+--------+--------+--------------+
|MUNICIPIO     |UF |SALAS 3D|SALAS 2D|TOTAL DE SALAS|
+--------------+---+--------+--------+--------------+
|SAO PAULO     |SP |167     |185,00  |352           |
|RIO DE JANEIRO|RJ |94      |122,00  |216           |
|BRASILIA      |DF |32      |56,00   |88            |
|BELO HORIZONTE|MG |33      |53,00   |86            |
|CURITIBA      |PR |46      |45,00   |91            |
+--------------+---+--------+--------+--------------+
only showing top 5 rows
root
 |-- MUNICIPIO: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- SALAS 3D: integer (nullable = true)
 |-- SALAS 2D: string (nullable = true)
 |-- TOTAL DE SALAS: integer (nullable = true)

+--------------+---+----------+--------------+-----+
|MUNICIPIO     |UF |PUBLICO   |RENDA         |PMI  |
+--------------+---+----------+--------------+-----+
|SAO PAULO     |SP |22.006.748|441.721.191,63|20,07|
|RIO DE JANEIRO|RJ |15.038.158|267.344.373,33|17,78|
|BRASILIA      |

In [ ]:
# Conferir os prints

print(df_publico.columns)
print(df_salas.columns)

['MUNICIPIO', 'UF', 'PUBLICO', 'RENDA', 'PMI']
['MUNICIPIO', 'UF', 'SALAS 3D', 'SALAS 2D', 'TOTAL DE SALAS']


In [ ]:
# Padronização de colunas

df_salas = df_salas \
    .withColumnRenamed("SALAS 3D", "SALAS_3D") \
    .withColumnRenamed("SALAS 2D", "SALAS_2D") \
    .withColumnRenamed("TOTAL DE SALAS", "TOTAL_DE_SALAS")

In [ ]:
# Conferir a padronização das colunas

print(df_salas.columns)

['MUNICIPIO', 'UF', 'SALAS_3D', 'SALAS_2D', 'TOTAL_DE_SALAS']


In [ ]:
# Limpeza de números

from pyspark.sql.functions import regexp_replace, col

df_publico = df_publico \
    .withColumn("PUBLICO", regexp_replace(col("PUBLICO"), r"\.", "")) \
    .withColumn("PUBLICO", col("PUBLICO").cast("long")) \
    .withColumn("RENDA", regexp_replace(col("RENDA"), r"\.", "")) \
    .withColumn("RENDA", regexp_replace(col("RENDA"), ",", ".")) \
    .withColumn("RENDA", col("RENDA").cast("double")) \
    .withColumn("PMI", regexp_replace(col("PMI"), ",", ".")) \
    .withColumn("PMI", col("PMI").cast("double"))

df_salas = df_salas \
    .withColumn("SALAS_2D", regexp_replace(col("SALAS_2D"), ",", ".")) \
    .withColumn("SALAS_2D", col("SALAS_2D").cast("double"))

In [ ]:
# Conferir a limpeza

df_publico.printSchema()
df_salas.printSchema()

df_publico.show(5)
df_salas.show(5)

root
 |-- MUNICIPIO: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- PUBLICO: long (nullable = true)
 |-- RENDA: double (nullable = true)
 |-- PMI: double (nullable = true)

root
 |-- MUNICIPIO: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- SALAS_3D: integer (nullable = true)
 |-- SALAS_2D: double (nullable = true)
 |-- TOTAL_DE_SALAS: integer (nullable = true)

+--------------+---+--------+--------------+-----+
|     MUNICIPIO| UF| PUBLICO|         RENDA|  PMI|
+--------------+---+--------+--------------+-----+
|     SAO PAULO| SP|22006748|4.4172119163E8|20.07|
|RIO DE JANEIRO| RJ|15038158|2.6734437333E8|17.78|
|      BRASILIA| DF| 5399645|  9.42073687E7|17.45|
|BELO HORIZONTE| MG| 5152266| 7.841315233E7|15.22|
|      CURITIBA| PR| 4323021| 7.342649313E7|16.98|
+--------------+---+--------+--------------+-----+
only showing top 5 rows
+--------------+---+--------+--------+--------------+
|     MUNICIPIO| UF|SALAS_3D|SALAS_2D|TOTAL_DE_SALAS|
+---------

In [ ]:
# Salvar os dataframes tratados em formato parquet

df_salas.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_FILMEB/dados_tratados/filmeb_salas_tratado")

df_publico.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_FILMEB/dados_tratados/filmeb_publico_tratado")

In [ ]:
# Leitura CSV ANCINE

df_bilheteria_dezembro = spark.read.csv(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_ANCINE/BilheteriaDezembro2025.csv",header=True,inferSchema=True)

df_bilheteria_janeiro = spark.read.csv(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_ANCINE/BilheteriaJaneiro2026.csv",header=True,inferSchema=True)

df_bilheteria_fevereiro = spark.read.csv(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_ANCINE/BilheteriaFevereiro2026.csv",header=True,inferSchema=True)

In [ ]:
# Conferir CSV

df_bilheteria_dezembro.show(5, truncate=False)
df_bilheteria_dezembro.printSchema()

df_bilheteria_janeiro.show(5, truncate=False)
df_bilheteria_janeiro.printSchema()

df_bilheteria_fevereiro.show(5, truncate=False)
df_bilheteria_fevereiro.printSchema()

+-------------+--------------------------------+-------------+--------------+---------+-------------+---------------------------+-------+-----------------------+-----------------+-----------------+---------------------+---+-----------------------------+
|DATA_EXIBICAO|TITULO_ORIGINAL                 |TITULO_BRASIL|CPB_ROE       |PAIS_OBRA|REGISTRO_SALA|NOME_SALA                  |PUBLICO|REGISTRO_GRUPO_EXIBIDOR|REGISTRO_EXIBIDOR|REGISTRO_COMPLEXO|MUNICIPIO            |UF |DISTRIBUIDORA                |
+-------------+--------------------------------+-------------+--------------+---------+-------------+---------------------------+-------+-----------------------+-----------------+-----------------+---------------------+---+-----------------------------+
|01/12/2025   |#SALVEROSA                      |NULL         |B2500337500000|BRASIL   |5006040      |CINEMARK FLAMBOYANT SALA 02|1      |6000018                |1843             |40181            |GOIANIA              |GO |ELO AUDIOVISUAL

In [ ]:
# Conversão de data string para tipo date - Dezembro

from pyspark.sql.functions import to_date, year, month

df_bilheteria_dezembro = df_bilheteria_dezembro.withColumn(
    "DATA_EXIBICAO",
    to_date("DATA_EXIBICAO", "dd/MM/yyyy")
).withColumn(
    "ANO", year("DATA_EXIBICAO")
).withColumn(
    "MES", month("DATA_EXIBICAO")
)

In [ ]:
# Conversão de data string para tipo date - Janeiro

from pyspark.sql.functions import to_date, year, month

df_bilheteria_janeiro = df_bilheteria_janeiro.withColumn(
    "DATA_EXIBICAO",
    to_date("DATA_EXIBICAO", "dd/MM/yyyy")
).withColumn(
    "ANO", year("DATA_EXIBICAO")
).withColumn(
    "MES", month("DATA_EXIBICAO")
)

In [ ]:
# Conversão de data string para tipo date - Fevereiro

from pyspark.sql.functions import to_date, year, month

df_bilheteria_fevereiro = df_bilheteria_fevereiro.withColumn(
    "DATA_EXIBICAO",
    to_date("DATA_EXIBICAO", "dd/MM/yyyy")
).withColumn(
    "ANO", year("DATA_EXIBICAO")
).withColumn(
    "MES", month("DATA_EXIBICAO")
)

In [ ]:
# Verificação da conversão das datas

df_bilheteria_dezembro.show(5, truncate=False)
df_bilheteria_dezembro.printSchema()

df_bilheteria_janeiro.show(5, truncate=False)
df_bilheteria_janeiro.printSchema()

df_bilheteria_fevereiro.show(5, truncate=False)
df_bilheteria_fevereiro.printSchema()

+-------------+--------------------------------+-------------+--------------+---------+-------------+---------------------------+-------+-----------------------+-----------------+-----------------+---------------------+---+-----------------------------+----+---+
|DATA_EXIBICAO|TITULO_ORIGINAL                 |TITULO_BRASIL|CPB_ROE       |PAIS_OBRA|REGISTRO_SALA|NOME_SALA                  |PUBLICO|REGISTRO_GRUPO_EXIBIDOR|REGISTRO_EXIBIDOR|REGISTRO_COMPLEXO|MUNICIPIO            |UF |DISTRIBUIDORA                |ANO |MES|
+-------------+--------------------------------+-------------+--------------+---------+-------------+---------------------------+-------+-----------------------+-----------------+-----------------+---------------------+---+-----------------------------+----+---+
|2025-12-01   |#SALVEROSA                      |NULL         |B2500337500000|BRASIL   |5006040      |CINEMARK FLAMBOYANT SALA 02|1      |6000018                |1843             |40181            |GOIANIA       

In [ ]:
# Juntar os CSVS em um único DataFrame

df_ancine = df_bilheteria_dezembro.unionByName(df_bilheteria_janeiro).unionByName(df_bilheteria_fevereiro)

In [ ]:
# Verificar junção

df_ancine.count()

575538

In [ ]:
# Verificar colunas

df_ancine.printSchema()
df_ancine.show(5, truncate=False)
df_ancine.count()

root
 |-- DATA_EXIBICAO: date (nullable = true)
 |-- TITULO_ORIGINAL: string (nullable = true)
 |-- TITULO_BRASIL: string (nullable = true)
 |-- CPB_ROE: string (nullable = true)
 |-- PAIS_OBRA: string (nullable = true)
 |-- REGISTRO_SALA: integer (nullable = true)
 |-- NOME_SALA: string (nullable = true)
 |-- PUBLICO: integer (nullable = true)
 |-- REGISTRO_GRUPO_EXIBIDOR: integer (nullable = true)
 |-- REGISTRO_EXIBIDOR: integer (nullable = true)
 |-- REGISTRO_COMPLEXO: integer (nullable = true)
 |-- MUNICIPIO: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- DISTRIBUIDORA: string (nullable = true)
 |-- ANO: integer (nullable = true)
 |-- MES: integer (nullable = true)

+-------------+--------------------------------+-------------+--------------+---------+-------------+---------------------------+-------+-----------------------+-----------------+-----------------+---------------------+---+-----------------------------+----+---+
|DATA_EXIBICAO|TITULO_ORIGINAL           

575538

In [ ]:
# Verificar valores nulos no DF Ancine

from pyspark.sql.functions import col, sum

df_ancine.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_ancine.columns
]).show()

+-------------+---------------+-------------+-------+---------+-------------+---------+-------+-----------------------+-----------------+-----------------+---------+---+-------------+---+---+
|DATA_EXIBICAO|TITULO_ORIGINAL|TITULO_BRASIL|CPB_ROE|PAIS_OBRA|REGISTRO_SALA|NOME_SALA|PUBLICO|REGISTRO_GRUPO_EXIBIDOR|REGISTRO_EXIBIDOR|REGISTRO_COMPLEXO|MUNICIPIO| UF|DISTRIBUIDORA|ANO|MES|
+-------------+---------------+-------------+-------+---------+-------------+---------+-------+-----------------------+-----------------+-----------------+---------+---+-------------+---+---+
|            0|              0|        61397|      0|        0|         1118|        0|      0|                  57804|             1118|             1118|        0|  0|            0|  0|  0|
+-------------+---------------+-------------+-------+---------+-------------+---------+-------+-----------------------+-----------------+-----------------+---------+---+-------------+---+---+



In [ ]:
# Verificar as linhas nulas antes de removê-las

df_ancine.filter(
    col("TITULO_BRASIL").isNull() & col("TITULO_ORIGINAL").isNull()
).show(5, truncate=False)

+-------------+---------------+-------------+-------+---------+-------------+---------+-------+-----------------------+-----------------+-----------------+---------+---+-------------+---+---+
|DATA_EXIBICAO|TITULO_ORIGINAL|TITULO_BRASIL|CPB_ROE|PAIS_OBRA|REGISTRO_SALA|NOME_SALA|PUBLICO|REGISTRO_GRUPO_EXIBIDOR|REGISTRO_EXIBIDOR|REGISTRO_COMPLEXO|MUNICIPIO|UF |DISTRIBUIDORA|ANO|MES|
+-------------+---------------+-------------+-------+---------+-------------+---------+-------+-----------------------+-----------------+-----------------+---------+---+-------------+---+---+
+-------------+---------------+-------------+-------+---------+-------------+---------+-------+-----------------------+-----------------+-----------------+---------+---+-------------+---+---+



In [ ]:
# Criar coluna Título Principal

from pyspark.sql.functions import col, coalesce

df_ancine = df_ancine.withColumn(
    "TITULO_PRINCIPAL",
    coalesce(col("TITULO_BRASIL"), col("TITULO_ORIGINAL"))
)

In [ ]:
# Remover linhas vazias

df_ancine = df_ancine.dropna(subset=[
    "TITULO_PRINCIPAL",
    "DATA_EXIBICAO",
    "PUBLICO"
])

In [ ]:
# Deixar apenas Título Principal

df_ancine = df_ancine.drop("TITULO_BRASIL", "TITULO_ORIGINAL")

In [ ]:
# Verificar esses ajustes

df_ancine.printSchema()
df_ancine.show(5, truncate=False)
df_ancine.count()

root
 |-- DATA_EXIBICAO: date (nullable = true)
 |-- CPB_ROE: string (nullable = true)
 |-- PAIS_OBRA: string (nullable = true)
 |-- REGISTRO_SALA: integer (nullable = true)
 |-- NOME_SALA: string (nullable = true)
 |-- PUBLICO: integer (nullable = true)
 |-- REGISTRO_GRUPO_EXIBIDOR: integer (nullable = true)
 |-- REGISTRO_EXIBIDOR: integer (nullable = true)
 |-- REGISTRO_COMPLEXO: integer (nullable = true)
 |-- MUNICIPIO: string (nullable = true)
 |-- UF: string (nullable = true)
 |-- DISTRIBUIDORA: string (nullable = true)
 |-- ANO: integer (nullable = true)
 |-- MES: integer (nullable = true)
 |-- TITULO_PRINCIPAL: string (nullable = true)

+-------------+--------------+---------+-------------+---------------------------+-------+-----------------------+-----------------+-----------------+---------------------+---+-----------------------------+----+---+--------------------------------+
|DATA_EXIBICAO|CPB_ROE       |PAIS_OBRA|REGISTRO_SALA|NOME_SALA                  |PUBLICO|REGISTRO_

575538

In [ ]:
# Salvar dataframe final Ancine

df_ancine.write.mode("overwrite").parquet(
    "/content/drive/MyDrive/ProjetoAnaliseCinema/dados_tratados/ancine_tratado")

In [ ]:
# Análise público por UF

df_publico_uf = df_ancine.groupBy("UF") \
    .sum("PUBLICO") \
    .withColumnRenamed("sum(PUBLICO)", "PUBLICO_TOTAL") \
    .orderBy("PUBLICO_TOTAL", ascending=False)

df_publico_uf.show()

+---+-------------+
| UF|PUBLICO_TOTAL|
+---+-------------+
| SP|      9991074|
| RJ|      3843564|
| MG|      2232370|
| PR|      1693470|
| RS|      1354864|
| SC|      1304204|
| BA|      1247302|
| PE|      1243089|
| CE|      1062471|
| DF|      1022104|
| GO|       997755|
| PA|       791938|
| AM|       600202|
| ES|       544090|
| MT|       485448|
| MA|       426753|
| RN|       421860|
| PB|       354475|
| AL|       337706|
| MS|       308427|
+---+-------------+
only showing top 20 rows


In [ ]:
# Análise salas por UF

df_salas_uf = df_ancine.select("UF", "REGISTRO_SALA") \
    .distinct() \
    .groupBy("UF") \
    .count() \
    .withColumnRenamed("count", "QTD_SALAS") \
    .orderBy("QTD_SALAS", ascending=False)

df_salas_uf.show()

+---+---------+
| UF|QTD_SALAS|
+---+---------+
| SP|     1119|
| RJ|      389|
| MG|      283|
| PR|      234|
| RS|      190|
| SC|      161|
| GO|      143|
| BA|      141|
| PE|      119|
| CE|      118|
| DF|       94|
| PA|       85|
| ES|       75|
| MT|       71|
| AM|       62|
| MA|       55|
| PB|       45|
| MS|       36|
| SE|       35|
| RN|       34|
+---+---------+
only showing top 20 rows


In [ ]:
# Juntar público por UF e salas por UF

df_analise_uf = df_publico_uf.join(df_salas_uf, on="UF")

In [ ]:
# Verificar junção público por UF e salas por UF

df_analise_uf.printSchema()
df_analise_uf.show()
df_analise_uf.count()

root
 |-- UF: string (nullable = true)
 |-- PUBLICO_TOTAL: long (nullable = true)
 |-- QTD_SALAS: long (nullable = false)

+---+-------------+---------+
| UF|PUBLICO_TOTAL|QTD_SALAS|
+---+-------------+---------+
| SC|      1304204|      161|
| RO|       154112|       30|
| PI|       249474|       27|
| AM|       600202|       62|
| RR|       108974|       17|
| GO|       997755|      143|
| TO|       129144|       26|
| MT|       485448|       71|
| SP|      9991074|     1119|
| ES|       544090|       75|
| PB|       354475|       45|
| RS|      1354864|      190|
| MS|       308427|       36|
| AL|       337706|       32|
| MG|      2232370|      283|
| PA|       791938|       85|
| BA|      1247302|      141|
| SE|       259175|       35|
| PE|      1243089|      119|
| CE|      1062471|      118|
+---+-------------+---------+
only showing top 20 rows


27

In [ ]:
# Conferir valores nulos no df_analise_uf

from pyspark.sql.functions import sum, col

df_analise_uf.select([
    sum(col(c).isNull().cast("int")).alias(c)
    for c in df_analise_uf.columns
]).show()

+---+-------------+---------+
| UF|PUBLICO_TOTAL|QTD_SALAS|
+---+-------------+---------+
|  0|            0|        0|
+---+-------------+---------+



In [ ]:
# Remover linhas 0

from pyspark.sql.functions import col

df_analise_uf = df_analise_uf.filter(col("UF") != "0")

In [ ]:
# Conferir remoção

df_analise_uf.select("UF").distinct().show()

+---+
| UF|
+---+
| SC|
| RO|
| PI|
| AM|
| RR|
| GO|
| TO|
| MT|
| SP|
| ES|
| PB|
| RS|
| MS|
| AL|
| MG|
| PA|
| BA|
| SE|
| PE|
| CE|
+---+
only showing top 20 rows


In [ ]:
# Criar coluna público por sala

df_analise_uf = df_analise_uf.withColumn(
    "PUBLICO_POR_SALA",
    col("PUBLICO_TOTAL") / col("QTD_SALAS")
)

In [ ]:
# Ordenar a coluna público por sala

df_analise_uf.orderBy("PUBLICO_POR_SALA", ascending=False).show()

+---+-------------+---------+------------------+
| UF|PUBLICO_TOTAL|QTD_SALAS|  PUBLICO_POR_SALA|
+---+-------------+---------+------------------+
| RN|       421860|       34| 12407.64705882353|
| AP|       182104|       15|12140.266666666666|
| AC|        79974|        7|11424.857142857143|
| DF|      1022104|       94|10873.446808510638|
| AL|       337706|       32|        10553.3125|
| PE|      1243089|      119|10446.126050420167|
| RJ|      3843564|      389| 9880.627249357327|
| AM|       600202|       62| 9680.677419354839|
| PA|       791938|       85| 9316.917647058823|
| PI|       249474|       27| 9239.777777777777|
| CE|      1062471|      118|  9003.99152542373|
| SP|      9991074|     1119| 8928.573726541556|
| BA|      1247302|      141| 8846.113475177304|
| MS|       308427|       36| 8567.416666666666|
| SC|      1304204|      161|8100.6459627329195|
| MG|      2232370|      283| 7888.233215547703|
| PB|       354475|       45| 7877.222222222223|
| MA|       426753| 

In [ ]:
# Conferir as colunas do df_analise_uf

df_analise_uf.columns

['UF', 'PUBLICO_TOTAL', 'QTD_SALAS', 'PUBLICO_POR_SALA']

In [ ]:
# Evolução do público por mês

df_publico_mes = df_ancine.groupBy("MES") \
    .sum("PUBLICO") \
    .withColumnRenamed("sum(PUBLICO)", "PUBLICO_TOTAL") \
    .orderBy("MES")

df_publico_mes.show()

+---+-------------+
|MES|PUBLICO_TOTAL|
+---+-------------+
|  1|     12038059|
|  2|      8522409|
| 12|     10865655|
+---+-------------+



In [ ]:
# Público por UF em fevereiro

from pyspark.sql.functions import col

df_fev_uf = df_ancine.filter(col("MES") == 2) \
    .groupBy("UF") \
    .sum("PUBLICO") \
    .withColumnRenamed("sum(PUBLICO)", "PUBLICO_FEVEREIRO") \
    .orderBy("PUBLICO_FEVEREIRO", ascending=False)

df_fev_uf.show()

+---+-----------------+
| UF|PUBLICO_FEVEREIRO|
+---+-----------------+
| SP|          2761103|
| RJ|          1048942|
| MG|           584156|
| PR|           405095|
| RS|           404958|
| BA|           361882|
| PE|           353002|
| SC|           341045|
| DF|           283857|
| GO|           254092|
| CE|           253305|
| PA|           230785|
| AM|           163385|
| RN|           137628|
| ES|           128985|
| MT|           114159|
| MA|           111954|
| PB|           105703|
| AL|            95587|
| MS|            80239|
+---+-----------------+
only showing top 20 rows


In [ ]:
# Semana do Cinema (R$10)

df_semana_cinema = df_ancine.filter(
    (col("DATA_EXIBICAO") >= "2026-02-08") &
    (col("DATA_EXIBICAO") <= "2026-02-14")
)

In [ ]:
# Público na semana so Cinema (R$10)

df_semana_cinema.groupBy("UF") \
    .sum("PUBLICO") \
    .orderBy("sum(PUBLICO)", ascending=False) \
    .show()

+---+------------+
| UF|sum(PUBLICO)|
+---+------------+
| SP|      891445|
| RJ|      349242|
| MG|      193551|
| RS|      141303|
| PR|      123054|
| BA|      122461|
| SC|      116670|
| PE|      116157|
| DF|       98105|
| PA|       88121|
| GO|       86508|
| CE|       74724|
| AM|       57601|
| RN|       52850|
| ES|       37761|
| MT|       37605|
| PB|       37463|
| MA|       36318|
| AL|       34203|
| MS|       27858|
+---+------------+
only showing top 20 rows


In [ ]:
# Participação público da semana do cinema por UF

from pyspark.sql.functions import col, sum, round

# total da semana
total_semana = df_semana_cinema.agg(
    sum("PUBLICO").alias("TOTAL")
).collect()[0]["TOTAL"]

# participação
df_participacao = df_semana_cinema.groupBy("UF") \
    .sum("PUBLICO") \
    .withColumnRenamed("sum(PUBLICO)", "PUBLICO_SEMANA") \
    .withColumn(
        "PARTICIPACAO_%",
        round((col("PUBLICO_SEMANA") / total_semana) * 100, 2)
    ) \
    .orderBy("PARTICIPACAO_%", ascending=False)

df_participacao.show()

+---+--------------+--------------+
| UF|PUBLICO_SEMANA|PARTICIPACAO_%|
+---+--------------+--------------+
| SP|        891445|         31.55|
| RJ|        349242|         12.36|
| MG|        193551|          6.85|
| RS|        141303|           5.0|
| PR|        123054|          4.35|
| BA|        122461|          4.33|
| SC|        116670|          4.13|
| PE|        116157|          4.11|
| DF|         98105|          3.47|
| PA|         88121|          3.12|
| GO|         86508|          3.06|
| CE|         74724|          2.64|
| AM|         57601|          2.04|
| RN|         52850|          1.87|
| ES|         37761|          1.34|
| MT|         37605|          1.33|
| PB|         37463|          1.33|
| MA|         36318|          1.29|
| AL|         34203|          1.21|
| MS|         27858|          0.99|
+---+--------------+--------------+
only showing top 20 rows


In [ ]:
# Uma linha por municipio no df_salas

from pyspark.sql.functions import sum

df_salas_agg = df_salas.groupBy("UF", "MUNICIPIO") \
    .agg(
        sum("SALAS_2D").alias("SALAS_2D"),
        sum("SALAS_3D").alias("SALAS_3D")
    )

In [ ]:
# Uma linha por municipio o df_publico

df_publico_agg = df_publico.groupBy("UF", "MUNICIPIO") \
    .agg(
        sum("RENDA").alias("RENDA"),
        sum("PMI").alias("PMI")
    )

In [ ]:
# Join df_salas e df_publico

df_filmeb = df_publico_agg.join(
    df_salas_agg,
    on=["UF", "MUNICIPIO"],
    how="left"
)

In [ ]:
# Verificar join df_salas e df_publico

df_filmeb.show()
df_filmeb.count()

+---+--------------+--------------+-----+--------+--------+
| UF|     MUNICIPIO|         RENDA|  PMI|SALAS_2D|SALAS_3D|
+---+--------------+--------------+-----+--------+--------+
| PA|         BELEM| 3.760196608E7|15.94|    20.0|      14|
| SP|     SAO PAULO|4.4172119163E8|20.07|   185.0|     167|
| SP|      SOROCABA| 2.520712675E7|13.83|     9.0|      29|
| DF|      BRASILIA|  9.42073687E7|17.45|    56.0|      32|
| SP|      CAMPINAS| 5.688311915E7|16.91|    36.0|      21|
| GO|       GOIANIA| 4.161965319E7|16.26|    27.0|      20|
| RS|  PORTO ALEGRE| 5.333244532E7|15.03|    48.0|      22|
| SP|     GUARULHOS| 3.681420425E7|17.11|    19.0|      17|
| RJ|RIO DE JANEIRO|2.6734437333E8|17.78|   122.0|      94|
| BA|      SALVADOR| 7.171143744E7|17.03|    34.0|      34|
| AM|        MANAUS| 5.072709417E7|14.25|    41.0|      28|
| PR|      CURITIBA| 7.342649313E7|16.98|    45.0|      46|
| PE|        RECIFE| 5.949005965E7| 17.4|    26.0|      28|
| MG|BELO HORIZONTE| 7.841315233E7|15.22

15

In [ ]:
# Criar variável total de salas

from pyspark.sql.functions import col

df_filmeb = df_filmeb.withColumn(
    "TOTAL_SALAS",
    col("SALAS_2D") + col("SALAS_3D")
)

In [ ]:
# Verificar PMI e RENDA

df_filmeb.select("PMI", "RENDA").show()

+-----+--------------+
|  PMI|         RENDA|
+-----+--------------+
|15.94| 3.760196608E7|
|20.07|4.4172119163E8|
|13.83| 2.520712675E7|
|17.45|  9.42073687E7|
|16.91| 5.688311915E7|
|16.26| 4.161965319E7|
|15.03| 5.333244532E7|
|17.11| 3.681420425E7|
|17.78|2.6734437333E8|
|17.03| 7.171143744E7|
|14.25| 5.072709417E7|
|16.98| 7.342649313E7|
| 17.4| 5.949005965E7|
|15.22| 7.841315233E7|
|15.99| 6.863527532E7|
+-----+--------------+



In [ ]:
# Agrupar PMI por faixa de preço

from pyspark.sql.functions import when

df_filmeb = df_filmeb.withColumn(
    "FAIXA_PRECO",
    when(col("PMI") < 10, "Até 10")
    .when((col("PMI") >= 10) & (col("PMI") < 20), "10 a 20")
    .when((col("PMI") >= 20) & (col("PMI") < 30), "20 a 30")
    .otherwise("30+")
)

In [ ]:
# Analisar renda por faixa de preço

df_filmeb.groupBy("FAIXA_PRECO") \
    .sum("RENDA") \
    .orderBy("FAIXA_PRECO") \
    .show()

+-----------+---------------+
|FAIXA_PRECO|     sum(RENDA)|
+-----------+---------------+
|    10 a 20|1.01541376881E9|
|    20 a 30| 4.4172119163E8|
+-----------+---------------+



In [ ]:
# Fazer renda média faixa de preço

from pyspark.sql.functions import avg

df_filmeb.groupBy("FAIXA_PRECO") \
    .agg(avg("RENDA").alias("RENDA_MEDIA")) \
    .orderBy("FAIXA_PRECO") \
    .show()

+-----------+-------------------+
|FAIXA_PRECO|        RENDA_MEDIA|
+-----------+-------------------+
|    10 a 20|7.252955491499999E7|
|    20 a 30|     4.4172119163E8|
+-----------+-------------------+



In [ ]:
# Correlação PMI e Renda   - # perto de 0 -> fraca ; # perto de 1 -> alta (relação positiva)

df_filmeb.stat.corr("PMI", "RENDA")

0.7369398092381159

In [ ]:
# Renda por quantidade de salas

df_filmeb.stat.corr("TOTAL_SALAS", "RENDA")

0.9945883513760995

In [ ]:
# Sala vs público

from pyspark.sql.functions import sum

# Público por UF
df_publico_uf = df_ancine.groupBy("UF") \
    .agg(sum("PUBLICO").alias("PUBLICO_TOTAL"))

# Salas por UF
df_salas_uf = df_ancine.select("UF", "REGISTRO_SALA") \
    .distinct() \
    .groupBy("UF") \
    .count() \
    .withColumnRenamed("count", "QTD_SALAS")

# Join
df_corr_publico_uf = df_publico_uf.join(df_salas_uf, "UF")

# Correlação
df_corr_publico_uf.stat.corr("QTD_SALAS", "PUBLICO_TOTAL")

0.9963781884376239

In [ ]:
# Correlação PMI vs total de salas

df_filmeb.stat.corr("PMI", "TOTAL_SALAS")

0.6982574035640791

In [ ]:
# Semana do cinema vs estrutura

df_semana = df_participacao.join(
    df_analise_uf.select("UF", "QTD_SALAS"),
    on="UF"
)

df_semana.stat.corr("QTD_SALAS", "PARTICIPACAO_%")

0.992819449872775

In [ ]:
# Converter parquet Ancine para CSV

df = spark.read.parquet("/content/drive/MyDrive/ProjetoAnaliseCinema/dados_tratados/ancine_tratado")

df.coalesce(1).write.mode("overwrite").csv("/content/dados_csv", header=True)

In [ ]:
# Converter parquet Film B pra CSV

df = spark.read.parquet("/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_FILMEB/dados_tratados/filmeb_salas_tratado")
df = spark.read.parquet("/content/drive/MyDrive/ProjetoAnaliseCinema/CSV_FILMEB/dados_tratados/filmeb_publico_tratado")

df_filmeb.coalesce(1).write.mode("overwrite").csv("/content/filmeb_csv", header=True)


In [ ]:
spark.stop()